# Post-hoc validation: Run A and Run B
Group 25 · Noise Schedule Ablation

Scores the pretrained weights and each run's final weights (`final_gen.pth`) on 200 held-out validation pairs, using the training objective (mean absolute error between true and predicted noise), under that run's own schedule. Run after fine-tuning; nothing was chosen from it. Run A and Run B values are not comparable with each other.

In each Google account, run Setup, Validation set and Preflight, then only that account's run cell (A or B). Needs T4 GPU; `finetune.py` and `val_loss.py` in `/content` or in `My Drive/FoundationModel-Assignment/`; the pretrained checkpoint in `My Drive/DW/LSUI/`; and `runs/RunA/final_gen.pth` or `runs/RunB/final_gen.pth`.

## Setup

In [ ]:
import os, sys, subprocess, json, random, hashlib, shutil, time
import numpy as np, pandas as pd, torch
assert torch.cuda.is_available(), "No GPU"
print("torch", torch.__version__, "|", torch.cuda.get_device_name(0), "| python", sys.version.split()[0])

from google.colab import drive
drive.mount("/content/drive")

A_DIR = "/content/drive/MyDrive/FoundationModel-Assignment"
CKPT_GEN = "/content/drive/MyDrive/DW/LSUI/I390000_E78_gen.pth"
SCRIPT, VAL_SCRIPT = "/content/finetune.py", "/content/val_loss.py"
OUT_VAL = f"{A_DIR}/runs/PostHocVal"
os.makedirs(OUT_VAL, exist_ok=True)

for s in (SCRIPT, VAL_SCRIPT):
    if not os.path.exists(s) and os.path.exists(f"{A_DIR}/{os.path.basename(s)}"):
        shutil.copy(f"{A_DIR}/{os.path.basename(s)}", s)
    assert os.path.exists(s), f"missing {s}"
assert os.path.exists(CKPT_GEN) and os.path.getsize(CKPT_GEN) > 5e8, f"checkpoint missing: {CKPT_GEN}"
CKPT_PREFIX = CKPT_GEN[:-len("_gen.pth")]

REPO_ROOT = "/content/DiffWater"; REPO_DIR = f"{REPO_ROOT}/DiffWater"
if not os.path.isdir(REPO_ROOT):
    subprocess.run(["git", "clone", "-q", "https://github.com/Guan-MS/DiffWater.git", REPO_ROOT], check=True)
subprocess.run(["git", "-C", REPO_ROOT, "checkout", "-q", "7db68f34060bbf1d7c5b8153664fb16a3dcda1ae"], check=True)
!pip -q install tensorboardX scikit-image lmdb pyyaml kagglehub pandas
!pip list 2>/dev/null | grep -i -E "^(scikit-image|numpy|pillow|pandas|torch) "

## Validation set

In [ ]:
from PIL import Image
import kagglehub

EUVP = f"{kagglehub.dataset_download('pamuduranasinghe/euvp-dataset')}/EUVP"
SEED, N_TEST, N_TRAIN, N_VAL, HAM = 1234, 30, 800, 200, 6
EXT = (".jpg", ".jpeg", ".png", ".bmp")

index = lambda d: {os.path.splitext(f)[0]: os.path.join(d, f) for f in sorted(os.listdir(d)) if f.lower().endswith(EXT)}
md5 = lambda p: hashlib.md5(open(p, "rb").read()).hexdigest()
save256 = lambda s, d: Image.open(s).convert("RGB").resize((256, 256), Image.Resampling.LANCZOS).save(d)
def dhash(path):
    a = np.asarray(Image.open(path).convert("L").resize((9, 8), Image.Resampling.LANCZOS), dtype=np.int16)
    return (a[:, 1:] > a[:, :-1]).ravel()

pairs = []
for sub in ("underwater_dark", "underwater_scenes"):
    a, b = index(f"{EUVP}/Paired/{sub}/trainA"), index(f"{EUVP}/Paired/{sub}/trainB")
    stems = sorted(set(a) & set(b))
    assert len(stems) > 0.9 * min(len(a), len(b)), "pairing by filename failed"
    pairs += [(sub, s, a[s], b[s]) for s in stems]
fp = {p[2]: (md5(p[2]), dhash(p[2])) for p in pairs}

dark = [p for p in pairs if p[0] == "underwater_dark"]
random.Random(SEED).shuffle(dark)
test, seen = [], set()
for p in dark:
    if fp[p[2]][0] in seen: continue
    seen.add(fp[p[2]][0]); test.append(p)
    if len(test) == N_TEST: break
test = sorted(test, key=lambda p: (p[0], p[1]))

test_md5 = {fp[p[2]][0] for p in test}; test_keys = {p[2] for p in test}
test_h = np.stack([fp[p[2]][1] for p in test])
pool = []
for p in pairs:
    if p[2] in test_keys: continue
    m, h = fp[p[2]]
    if m in test_md5 or (test_h != h).sum(axis=1).min() <= HAM: continue
    pool.append(p)
random.Random(SEED).shuffle(pool)
train, rest = pool[:N_TRAIN], pool[N_TRAIN:]

split_fp = hashlib.md5(json.dumps([[p[0], p[1]] for p in test]).encode()).hexdigest()[:12]
train_fp = hashlib.md5("".join(sorted(f"{i:05d}_{sub}_{s}.png" for i, (sub, s, _, _) in enumerate(train))).encode()).hexdigest()[:12]
print("split fingerprint", split_fp, "| train fingerprint", train_fp)
assert split_fp == "f8cc4714a30c" and train_fp == "6a9f33de995c", "split differs from the experiment"

train_md5 = {fp[p[2]][0] for p in train}
train_h = np.stack([fp[p[2]][1] for p in train])
val, seen_v, n_exact, n_near = [], set(), 0, 0
for p in rest:
    m, h = fp[p[2]]
    if m in train_md5 or m in seen_v: n_exact += 1; continue
    if (train_h != h).sum(axis=1).min() <= HAM: n_near += 1; continue
    seen_v.add(m); val.append(p)
    if len(val) == N_VAL: break
assert len(val) == N_VAL
key = lambda p: (p[0], p[1])
assert not ({key(p) for p in val} & ({key(p) for p in train} | {key(p) for p in test})), "validation overlaps training or test"
print(f"validation {len(val)} pairs (dark {sum(p[0] == 'underwater_dark' for p in val)}, scenes {sum(p[0] == 'underwater_scenes' for p in val)}); "
      f"skipped {n_exact} exact, {n_near} near-duplicates of training images")

OUT_VAL_SET = "/content/euvp_val"
shutil.rmtree(OUT_VAL_SET, ignore_errors=True)
for sub in ("input_256", "target_256"): os.makedirs(f"{OUT_VAL_SET}/{sub}")
for i, (sub, s, pa, pb) in enumerate(val):
    save256(pa, f"{OUT_VAL_SET}/input_256/{i:04d}_{sub}_{s}.png"); save256(pb, f"{OUT_VAL_SET}/target_256/{i:04d}_{sub}_{s}.png")
vnames = sorted(os.listdir(f"{OUT_VAL_SET}/input_256"))
val_fp = hashlib.md5("".join(vnames).encode()).hexdigest()[:12]
json.dump([[p[0], p[1]] for p in val], open(f"{OUT_VAL}/val_split.json", "w"))
print("val fingerprint", val_fp, "(must match in both accounts)")

## Preflight

In [ ]:
EXPECT = dict(finetune_md5="38c97203e8ccafdfb80cb2be74b0bb3a", val_loss_md5="74920bbc7d211bcb34f589b831e9ac1d",
              commit="7db68f34060bbf1d7c5b8153664fb16a3dcda1ae", train_fp="6a9f33de995c", split_fp="f8cc4714a30c")
got = dict(finetune_md5=md5(SCRIPT), val_loss_md5=md5(VAL_SCRIPT),
           commit=subprocess.run(["git", "-C", REPO_ROOT, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip(),
           train_fp=train_fp, split_fp=split_fp)
for k in EXPECT:
    print(f"{k:13s} {got[k]}  {'OK' if got[k] == EXPECT[k] else 'MISMATCH, expected ' + EXPECT[k]}")
assert all(got[k] == EXPECT[k] for k in EXPECT), "not the frozen experiment"
for r in ("A", "B"):
    f = f"{A_DIR}/runs/Run{r}/final_gen.pth"
    print(f"Run {r} final weights:", f"{os.path.getsize(f)/1e6:.1f} MB" if os.path.exists(f) else "not in this account")
PREFLIGHT_OK = True

## Run A account only

In [ ]:
RUN = "A"
FINAL = f"{A_DIR}/runs/RunA/final_gen.pth"
assert globals().get("PREFLIGHT_OK"), "run Preflight first"
assert os.path.exists(FINAL) and os.path.getsize(FINAL) > 5e8, f"final weights missing: {FINAL}"
assert not subprocess.run('pgrep -f "[v]al_loss.py"', shell=True, capture_output=True, text=True).stdout.strip(), "val_loss.py already running"

cmd = (f"exec python -u {VAL_SCRIPT} --run {RUN} --repo_dir {REPO_DIR} --ckpt_prefix {CKPT_PREFIX} "
       f"--final_ckpt {FINAL} --val_root {OUT_VAL_SET} --out_dir {OUT_VAL}")
print(cmd)
p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
try:
    for line in p.stdout:
        if "UserWarning" not in line and "lab2rgb" not in line:
            print(line, end="")
except KeyboardInterrupt:
    p.terminate(); raise
finally:
    rc = p.wait()
assert rc == 0, f"val_loss.py exited with code {rc}"

## Run B account only

In [ ]:
RUN = "B"
FINAL = f"{A_DIR}/runs/RunB/final_gen.pth"
assert globals().get("PREFLIGHT_OK"), "run Preflight first"
assert os.path.exists(FINAL) and os.path.getsize(FINAL) > 5e8, f"final weights missing: {FINAL}"
assert not subprocess.run('pgrep -f "[v]al_loss.py"', shell=True, capture_output=True, text=True).stdout.strip(), "val_loss.py already running"

cmd = (f"exec python -u {VAL_SCRIPT} --run {RUN} --repo_dir {REPO_DIR} --ckpt_prefix {CKPT_PREFIX} "
       f"--final_ckpt {FINAL} --val_root {OUT_VAL_SET} --out_dir {OUT_VAL}")
print(cmd)
p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
try:
    for line in p.stdout:
        if "UserWarning" not in line and "lab2rgb" not in line:
            print(line, end="")
except KeyboardInterrupt:
    p.terminate(); raise
finally:
    rc = p.wait()
assert rc == 0, f"val_loss.py exited with code {rc}"

## Collect

In [ ]:
rows = []
for r in ("A", "B"):
    f = f"{OUT_VAL}/val_summary_Run{r}.json"
    if not os.path.exists(f): continue
    s = json.load(open(f))
    tl = f"{A_DIR}/runs/Run{r}/train_log.csv"
    tr = pd.read_csv(tl) if os.path.exists(tl) else None
    rows.append(dict(run=r, n_val=s["n_val"], val_fp=s["val_fingerprint"], pretrained=round(s["val_loss_pretrained_mean"], 5),
                     finetuned=round(s["val_loss_finetuned_mean"], 5), diff=round(s["paired_diff_finetuned_minus_pretrained"], 5),
                     se=round(s["paired_diff_se"], 5), lower_on=f'{s["n_images_finetuned_lower"]}/{s["n_val"]}',
                     train_loss_ep10=None if tr is None else round(float(tr.train_loss.iloc[-1]), 5)))
    print(f"Run {r} per-timestep difference (fine-tuned minus pretrained)")
    print(pd.DataFrame(s["per_timestep"]).round(5).to_string(index=False))
print(pd.DataFrame(rows).to_string(index=False) if rows else "no summary in this account yet")
shutil.make_archive("/content/PostHocVal", "zip", OUT_VAL)
print("results in Drive:", OUT_VAL)